In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import ndcg_score

In [3]:
df = pd.read_csv("predictions_test_sample.csv")
df.head()

,cvid,jobid,score
0,d39deb03fc6b4b739cd153536a19fb45,1534557,0.971680
1,d39deb03fc6b4b739cd153536a19fb45,1530549,0.965820
2,d39deb03fc6b4b739cd153536a19fb45,1575598,0.805664
3,d39deb03fc6b4b739cd153536a19fb45,1588430,0.750977
4,d39deb03fc6b4b739cd153536a19fb45,1553824,0.746094


In [7]:
df_base = pd.read_csv("../dataset/final_dataset/contacted_anon.csv")
df_base["response"] = df_base["response"].fillna(0)

strict = False

if strict:
    df_base["response"] = df_base["response"].apply(lambda x: 1 if "ok" in str(x) else 0)
else:
    df_base["response"] = df_base["response"].apply(lambda x: x if x == 0 else 1)

df_scores = df_base[["cvid", "humanjobid", "response"]]
df_scores.head()

,cvid,humanjobid,response
0,971b2f2bbccb436dbf11215a8e5c9463,1527545.0,0
1,6652b22903a84153903b7e5fef960897,1527545.0,1
2,6a21e3c95b364fda8633413972e936f4,1527545.0,0
3,ecc450aa96f3489b9ade93a3a5c13423,1527545.0,0
4,c5692d7b3df74c1880b9f228fe6ed115,1527545.0,1


In [8]:
df_full = pd.merge(df, df_scores, left_on=["cvid", "jobid"], right_on=["cvid", "humanjobid"])
df_full = df_full.drop("humanjobid", axis=1)
df_full.head()

,cvid,jobid,score,response
0,d39deb03fc6b4b739cd153536a19fb45,1534557,0.971680,1
1,d39deb03fc6b4b739cd153536a19fb45,1530549,0.965820,0
2,d39deb03fc6b4b739cd153536a19fb45,1575598,0.805664,1
3,d39deb03fc6b4b739cd153536a19fb45,1588430,0.750977,1
4,d39deb03fc6b4b739cd153536a19fb45,1553824,0.746094,1


In [9]:
ndcgs = []
for group in df_full.groupby("cvid"):
    ndcgs.append(ndcg_score(np.array(group[1]["response"].values).reshape(1, -1), 
                            np.array(group[1]["score"].values).reshape(1, -1), k=10))

np.mean(ndcgs)

0.5315316262717783

In [18]:
def evaluate_dataframe_fairness(df, gender_map, area_map, k=10):
    """
    Evaluates NDCG and Fairness metrics on a precomputed DataFrame.
    Expected DataFrame columns: ['cvid', 'jobid', 'score', 'response']
    """
    # Track NDCG separately by gender
    ndcg_scores = {'Male': [], 'Female': [], 'Other': [], 'Unknown': []}
    all_scores = []
    
    # Track global counts for dataset-relative Disparate Visibility (ΔV)
    total_rural_dataset = 0
    total_items_dataset = 0
    total_rural_recommended = 0
    total_items_recommended = 0
    
    vacancy_pool_sizes = []

    # Group the dataframe by candidate
    grouped = df.groupby('cvid')

    for cvid, group in grouped:
        # ndcg_score expects 2D arrays of shape (1, n_items)
        y_true = group['response'].values.reshape(1, -1)
        y_score = group['score'].values.reshape(1, -1)
        
        # Calculate NDCG (skipping candidates with only 1 job as ranking is undefined)
        if y_true.shape[1] > 1:
            try:
                score = ndcg_score(y_true, y_score, k=k)
                all_scores.append(score)
            except ValueError:
                continue
        else:
            continue
            
        # --- Gender Fairness (Utility) ---
        gender = gender_map.get(cvid, 'Unknown')
        ndcg_scores[gender].append(score)
        
        # --- Geographic Fairness (ΔV) ---
        # Ensure job IDs are strings for dictionary matching
        vac_ids = group['jobid'].astype(str).tolist()
        vacancy_pool_sizes.append(len(vac_ids))
        
        if len(vac_ids) > 0:
            # Baseline: Accumulate Rural items across the dataset pool
            batch_rural_count = sum(1 for v in vac_ids if area_map.get(int(float(v))) == 'Rural')
            total_rural_dataset += batch_rural_count
            total_items_dataset += len(vac_ids)
            
            # Recommendations: Accumulate Rural items in the Top-K recommendations
            actual_k = min(k, len(vac_ids))
            if actual_k > 0:
                # Use pandas nlargest to quickly grab the top K scored rows
                top_k_group = group.nlargest(actual_k, 'score')
                top_k_vacs = top_k_group['jobid'].astype(str).tolist()
                
                top_k_rural_count = sum(1 for v in top_k_vacs if area_map.get(int(float(v))) == 'Rural')
                total_rural_recommended += top_k_rural_count
                total_items_recommended += actual_k

    # --- Utility (Gender) Summary ---
    mean_male_ndcg = np.mean(ndcg_scores['Male']) if ndcg_scores['Male'] else 0.0
    mean_female_ndcg = np.mean(ndcg_scores['Female']) if ndcg_scores['Female'] else 0.0
    
    print(f"Overall Average NDCG: {np.mean(all_scores):.4f}")
    print(f"Female NDCG: {mean_female_ndcg:.4f} (n={len(ndcg_scores['Female'])}) | Male NDCG: {mean_male_ndcg:.4f} (n={len(ndcg_scores['Male'])})")

    if len(ndcg_scores['Male']) > 0 and len(ndcg_scores['Female']) > 0:
        ndcg_gap = mean_female_ndcg - mean_male_ndcg
        print(f"Performance Disparity : {ndcg_gap:.4f}\n")
    else:
        print("Performance Disparity : N/A (Missing demographic group)\n")
        ndcg_gap = None

    # --- Geographic Fairness Summary ---
    avg_pool = np.mean(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    min_pool = np.min(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    max_pool = np.max(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    
    print("--- Geographic Fairness Summary ---")
    print(f"Total Candidates          : {len(vacancy_pool_sizes)}")
    print(f"Vacancies per Candidate   : Mean: {avg_pool:.1f} | Min: {min_pool} | Max: {max_pool}")
    
    if total_items_dataset > 0 and total_items_recommended > 0:
        frac_dataset = total_rural_dataset / total_items_dataset
        frac_recommended = total_rural_recommended / total_items_recommended
        mean_disp_vis = frac_recommended - frac_dataset
        
        print(f"Rural Fraction in Dataset      : {frac_dataset * 100:.2f}%")
        print(f"Rural Fraction in Top-{k} Recoms : {frac_recommended * 100:.2f}%")
        print(f"Disparate Visibility (\u0394V)      : {mean_disp_vis:.4f}\n")
    else:
        print("Disparate Visibility (\u0394V): N/A (No valid vacancy pools)\n")
        mean_disp_vis = None
            
    return np.mean(all_scores), ndcg_gap, mean_disp_vis

In [19]:
df_gender = pd.read_csv("./fairness/anonid_gender_mapping.csv")
gender_map = dict(zip(df_gender['anon_id'], df_gender['gender']))

df_location = pd.read_csv("./fairness/job_area_mapping.csv")
area_map = dict(zip(df_location["humanjobid"], df_location["area"]))

evaluate_dataframe_fairness(df_full, gender_map, area_map)

Overall Average NDCG: 0.5315
Female NDCG: 0.5564 (n=22) | Male NDCG: 0.5207 (n=37)
Performance Disparity : -0.0358

--- Geographic Fairness Summary ---
Total Candidates          : 60
Vacancies per Candidate   : Mean: 18.4 | Min: 10 | Max: 41
Rural Fraction in Dataset      : 47.73%
Rural Fraction in Top-10 Recoms : 49.00%
Disparate Visibility (ΔV)      : 0.0127



(0.5315316262717783, -0.035753333347726346, 0.012686025408348456)